# Notebook for extracting mandatory subjects for all schools

In [8]:
from firecrawl_app import app
from pydantic import BaseModel
import os
import sys
import json
import pandas as pd

# Optional if you want to import other helpers
sys.path.append(os.path.abspath(".."))

# 🔸 Define a basic schema to match the output we want (just the mandatory subjects)
class DummySchema(BaseModel):
    obligatorise_emner: list[str]

# 🔁 Function to try multiple prompts
def try_multiple_prompts(url, prompts, app):
    for prompt in prompts:
        try:
            data = app.extract([url], {
                'prompt': prompt,
                'schema': DummySchema.model_json_schema()
            })
            subjects = data.get("data", {}).get("obligatorise_emner", [])
            if subjects:
                return subjects, prompt
        except Exception as e:
            print(f"Prompt failed: {prompt}\nError: {e}")
    return [], None

# 📂 Load config
filepath = "config_mandatory.json"
with open(filepath, "r", encoding="utf-8") as json_file:
    config_data = json.load(json_file)

# 📋 Output structure
first_write = True
output_file = "Mandatory_subjects.csv"
rows = []

# 🔁 Loop through each school and program
for school, school_data in config_data.items():
    print(f"\nNow extracting mandatory subjects from {school}")

    for study_program, url in school_data["mandatory_subjects"].items():
        print(f" - Extracting for program: {study_program}")
        
        try:
            prompts = school_data["prompt_mandatory_subjects"]
            if isinstance(prompts, str):
                prompts = [prompts]  # Ensure it's a list

            subjects, used_prompt = try_multiple_prompts(
                url=url,
                prompts=prompts,
                app=app
            )

            if subjects:
                for subject in subjects:
                    rows.append({
                        "school": school,
                        "study_program": study_program,
                        "type": "Obligatorise_emner",
                        "mandatory_subject": subject.strip()
                    })
            else:
                print("   ⚠️ No mandatory subjects found.")
                rows.append({
                    "school": school,
                    "study_program": study_program,
                    "type": "Obligatorise_emner",
                    "mandatory_subject": ""
                })

        except Exception as e:
            print(f"  ❌ Failed to extract for {study_program}: {e}")
            rows.append({
                "school": school,
                "study_program": study_program,
                "type": "Obligatorise_emner",
                "mandatory_subject": ""
            })

# 💾 Save result to CSV
df = pd.DataFrame(rows)
df.to_csv(output_file, index=False)
print("\n✅ Mandatory subjects written to CSV!")



Now extracting mandatory subjects from UIO
 - Extracting for program: Informatikk: Programmering og systemarkitektur
 - Extracting for program: Informatikk: Design, bruk og interaksjon
 - Extracting for program: Informatikk: Digital oekonomi og ledelse

Now extracting mandatory subjects from UIB
 - Extracting for program: Bachelor i informatikk, datateknologi
 - Extracting for program: Bachelor i kunstig intelligens
   ⚠️ No mandatory subjects found.

Now extracting mandatory subjects from UIA
 - Extracting for program: Bachelor i ingenioerfag, data

Now extracting mandatory subjects from NTNU
 - Extracting for program: Bachelor i programmering
 - Extracting for program: Bachelor i informatikk
   ⚠️ No mandatory subjects found.
 - Extracting for program: Bachelor i ingenioerfag, data
   ⚠️ No mandatory subjects found.

Now extracting mandatory subjects from HIOF
 - Extracting for program: Bachelor i ingeniørfag data, dataingienør

Now extracting mandatory subjects from KRISTIANIA
 - E

In [9]:
""" import os
import sys
from firecrawl_app import app, ExtractSchema, NestedModel
import json
import pandas as pd
sys.path.append(os.path.abspath(".."))

from utils.helpers import create_csv
from utils.helpers import *

filepath = "config_mandatory.json"

# Load the JSON file containing config
with open(filepath, "r", encoding="utf-8") as json_file:
    config_data = json.load(json_file)
    
# Get all schools
schools = list(config_data.keys())

print("Schools currently defined in config_mandatory.json:")
print("-" * 40)
for school in sorted(schools):
    print(f"- {school}")  """

' import os\nimport sys\nfrom firecrawl_app import app, ExtractSchema, NestedModel\nimport json\nimport pandas as pd\nsys.path.append(os.path.abspath(".."))\n\nfrom utils.helpers import create_csv\nfrom utils.helpers import *\n\nfilepath = "config_mandatory.json"\n\n# Load the JSON file containing config\nwith open(filepath, "r", encoding="utf-8") as json_file:\n    config_data = json.load(json_file)\n    \n# Get all schools\nschools = list(config_data.keys())\n\nprint("Schools currently defined in config_mandatory.json:")\nprint("-" * 40)\nfor school in sorted(schools):\n    print(f"- {school}")  '

## Code for running extraction of mandatory subjects for all schools defined in config_mandatory.json.

In [10]:
first_write = True
output_file = "Second_Mandatory_subjects.csv"
rows = []

for school, school_data in config_data.items():
    print(f"\nNow extracting mandatory subjects from {school}")

    for study_program, url in school_data["mandatory_subjects"].items():
        print(f" - Extracting for program: {study_program}")
        
        try:
            data = app.extract([url], {
                'prompt': school_data["prompt_mandatory_subjects"],
                'schema': ExtractSchema.model_json_schema()
            })

            subjects = data["data"]["læringsutbyttebeskrivelser"].get("obligatorise_emner", [])

            if subjects:
                for subject in subjects:
                    rows.append({
                        "school": school,
                        "study_program": study_program,
                        "type": "Obligatorise_emner",
                        "mandatory_subject": subject.strip()
                    })
            else:
                print("   ⚠️ No mandatory subjects found.")
                rows.append({
                    "school": school,
                    "study_program": study_program,
                    "type": "Obligatorise_emner",
                    "mandatory_subject": ""
                })

        except Exception as e:
            print(f"  ❌ Failed to extract for {study_program}: {e}")
            # Log failure as an empty row too
            rows.append({
                "school": school,
                "study_program": study_program,
                "type": "Obligatorise_emner",
                "mandatory_subject": ""
            })

# Save all collected rows
df = pd.DataFrame(rows)
df.to_csv(output_file, index=False)
print("\n✅ Mandatory subjects written to CSV!")



Now extracting mandatory subjects from UIO
 - Extracting for program: Informatikk: Programmering og systemarkitektur
 - Extracting for program: Informatikk: Design, bruk og interaksjon
 - Extracting for program: Informatikk: Digital oekonomi og ledelse

Now extracting mandatory subjects from UIB
 - Extracting for program: Bachelor i informatikk, datateknologi
 - Extracting for program: Bachelor i kunstig intelligens

Now extracting mandatory subjects from UIA
 - Extracting for program: Bachelor i ingenioerfag, data
  ❌ Failed to extract for Bachelor i ingenioerfag, data: ("Unexpected error during extract: Status code 400. Bad Request - [{'code': 'invalid_type', 'expected': 'string', 'received': 'array', 'path': ['prompt'], 'message': 'Expected string, received array'}]", 500)

Now extracting mandatory subjects from NTNU
 - Extracting for program: Bachelor i programmering
  ❌ Failed to extract for Bachelor i programmering: ("Unexpected error during extract: Status code 400. Bad Request